# Silver - Limpieza y enriquecimiento

Parte 3 del proyecto: leer Bronze, tipar datos, eliminar duplicados y generar variables derivadas por ventanas temporales.

In [43]:
import os
import socket
import time
from urllib import error, request
from py4j.protocol import Py4JJavaError
from pyspark.sql import SparkSession


def resolve_service_host(service_name: str, fallback_host: str = "localhost") -> str:
    try:
        socket.gethostbyname(service_name)
        return service_name
    except socket.gaierror:
        return fallback_host


def is_http_ready(url: str, timeout: float = 2.0) -> bool:
    try:
        with request.urlopen(url, timeout=timeout) as _:
            return True
    except error.HTTPError:
        return True
    except Exception:
        return False


iceberg_host = resolve_service_host("iceberg-rest")
minio_host = resolve_service_host("minio")

default_iceberg_uri = f"http://{iceberg_host}:8181"
default_minio_endpoint = f"http://{minio_host}:{'9000' if minio_host == 'minio' else '9100'}"

ICEBERG_REST_URI = os.getenv("ICEBERG_REST_URI", default_iceberg_uri)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", default_minio_endpoint)

print(f"ICEBERG_REST_URI={ICEBERG_REST_URI}")
print(f"MINIO_ENDPOINT={MINIO_ENDPOINT}")

iceberg_config_url = f"{ICEBERG_REST_URI}/v1/config"
for _ in range(20):
    if is_http_ready(iceberg_config_url):
        break
    time.sleep(2)

spark = (
    SparkSession.builder
    .appName("silver-payments-job")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.6.1"
        ])
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", ICEBERG_REST_URI)
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", MINIO_ENDPOINT)
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "admin123")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

for attempt in range(1, 11):
    try:
        spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.silver")
        print("Spark e Iceberg listos")
        break
    except Py4JJavaError as e:
        message = str(e)
        transient = "UnknownHostException" in message or "RESTException" in message
        if transient and attempt < 10:
            print(f"Reintento {attempt}/10: esperando catalogo Iceberg...")
            time.sleep(3)
            continue
        raise

ICEBERG_REST_URI=http://iceberg-rest:8181
MINIO_ENDPOINT=http://minio:9000
Spark e Iceberg listos


## 1) Lectura de Bronze y conversión de tipos

In [44]:
from pyspark.sql import functions as F

bronze_df = spark.table("lakehouse.bronze.payments_raw")

typed_df = (
    bronze_df
    .withColumn("event_ts", F.to_timestamp("event_time"))
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn("country", F.upper(F.col("country")))
    .withColumn("currency", F.upper(F.col("currency")))
    .withColumn("status", F.lower(F.col("status")))
)

typed_df.select("event_time", "event_ts", "amount", "country", "currency", "status").show(5, truncate=False)

+---------------------------+--------------------------+------+-------+--------+--------+
|event_time                 |event_ts                  |amount|country|currency|status  |
+---------------------------+--------------------------+------+-------+--------+--------+
|2026-03-16T09:12:08.312735Z|2026-03-16 09:12:08.312735|102.70|TH     |EUR     |approved|
|2026-02-04T20:56:39.343849Z|2026-02-04 20:56:39.343849|205.60|CZ     |AUD     |approved|
|2026-02-02T02:39:40.444544Z|2026-02-02 02:39:40.444544|99.58 |AR     |EUR     |approved|
|2026-03-03T05:29:14.545384Z|2026-03-03 05:29:14.545384|127.59|HT     |CAD     |approved|
|2026-02-11T22:45:21.646152Z|2026-02-11 22:45:21.646152|131.11|PA     |CAD     |approved|
+---------------------------+--------------------------+------+-------+--------+--------+
only showing top 5 rows



## 2) Eliminación de duplicados por payment_id (último evento)

In [45]:
from pyspark.sql.window import Window

w_dedup = Window.partitionBy("payment_id").orderBy(F.col("event_ts").desc_nulls_last())

dedup_df = (
    typed_df
    .withColumn("rn", F.row_number().over(w_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print(f"Registros bronze: {bronze_df.count()}")
print(f"Registros tras deduplicar: {dedup_df.count()}")

Registros bronze: 65960
Registros tras deduplicar: 65960


## 3) Enriquecimiento con ventanas temporales

In [46]:
base = dedup_df.withColumn("event_ts_seconds", F.col("event_ts").cast("long"))

w_card_5m = Window.partitionBy("card_id").orderBy("event_ts_seconds").rangeBetween(-300, 0)
w_card_10m = Window.partitionBy("card_id").orderBy("event_ts_seconds").rangeBetween(-600, 0)
w_card_1h = Window.partitionBy("card_id").orderBy("event_ts_seconds").rangeBetween(-3600, 0)
w_device_1h = Window.partitionBy("device_id").orderBy("event_ts_seconds").rangeBetween(-3600, 0)

silver_df = (
    base
    .withColumn("tx_count_card_5m", F.count("payment_id").over(w_card_5m))
    .withColumn("distinct_merchants_card_10m", F.approx_count_distinct("merchant_id").over(w_card_10m))
    .withColumn("distinct_countries_card_1h", F.approx_count_distinct("country").over(w_card_1h))
    .withColumn("distinct_cards_device_1h", F.approx_count_distinct("card_id").over(w_device_1h))
    .withColumn(
        "declined_ratio_card_1h",
        F.avg(F.when(F.col("status") == "declined", F.lit(1.0)).otherwise(F.lit(0.0))).over(w_card_1h)
    )
    .drop("event_ts_seconds")
)

silver_df.select(
    "payment_id",
    "card_id",
    "event_ts",
    "tx_count_card_5m",
    "distinct_merchants_card_10m",
    "distinct_countries_card_1h",
    "distinct_cards_device_1h",
    "declined_ratio_card_1h"
).show(10, truncate=False)

+--------------------------------------------+----------+--------------------------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|payment_id                                  |card_id   |event_ts                  |tx_count_card_5m|distinct_merchants_card_10m|distinct_countries_card_1h|distinct_cards_device_1h|declined_ratio_card_1h|
+--------------------------------------------+----------+--------------------------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|75223bb2-0a69-40bf-9348-4f42559b1c0a        |CARD_00001|2026-01-13 19:26:46.153016|1               |1                          |1                         |1                       |0.0                   |
|74feba15-f02e-491e-a4c8-9e99fad58ab0        |CARD_00001|2026-01-13 21:39:04.069087|1               |1                          |1                         |2                       

## 4) Escritura a Iceberg Silver

In [47]:
import time
from py4j.protocol import Py4JJavaError

target_table = "lakehouse.silver.payments_enriched"
expected_count = silver_df.count()
max_attempts = 3
last_error = None
unknown_commit_seen = False

for attempt in range(1, max_attempts + 1):
    try:
        (
            silver_df.writeTo(target_table)
            .using("iceberg")
            .tableProperty("write.format.default", "parquet")
            .createOrReplace()
        )
        print(f"Tabla Silver actualizada: {target_table} (intento {attempt})")
        last_error = None
        break
    except Py4JJavaError as e:
        last_error = e
        error_text = str(e)
        unknown_commit = (
            "CommitStateUnknownException" in error_text
            or "Cannot determine whether the commit was successful" in error_text
        )

        if unknown_commit:
            unknown_commit_seen = True
            if spark.catalog.tableExists(target_table):
                actual_count = spark.table(target_table).count()
                print(
                    f"Commit con estado desconocido. Tabla disponible con {actual_count} filas "
                    f"(esperado ~ {expected_count})."
                )
            else:
                print("Commit con estado desconocido y tabla aún no visible en catálogo.")

        if attempt == max_attempts:
            break

        wait_seconds = attempt * 2
        print(f"Fallo al escribir Silver (intento {attempt}): {e}")
        print(f"Reintentando en {wait_seconds}s...")
        time.sleep(wait_seconds)

if last_error is not None and not unknown_commit_seen:
    raise last_error

if last_error is not None and unknown_commit_seen:
    print(
        "Se detectó CommitStateUnknownException en todos los intentos. "
        "Continúa con la celda de validación para confirmar el estado final de la tabla."
    )

Tabla Silver actualizada: lakehouse.silver.payments_enriched (intento 1)


## 5) Validación rápida de resultados

In [48]:
spark.sql("SELECT COUNT(*) AS total_silver FROM lakehouse.silver.payments_enriched").show()

spark.sql("""
SELECT
    payment_id,
    event_ts,
    card_id,
    status,
    tx_count_card_5m,
    distinct_merchants_card_10m,
    distinct_countries_card_1h,
    distinct_cards_device_1h,
    ROUND(declined_ratio_card_1h, 4) AS declined_ratio_card_1h
FROM lakehouse.silver.payments_enriched
ORDER BY event_ts DESC
LIMIT 10
""").show(truncate=False)

+------------+
|total_silver|
+------------+
|       65960|
+------------+

+------------------------------------+--------------------------+----------+--------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|payment_id                          |event_ts                  |card_id   |status  |tx_count_card_5m|distinct_merchants_card_10m|distinct_countries_card_1h|distinct_cards_device_1h|declined_ratio_card_1h|
+------------------------------------+--------------------------+----------+--------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|68268268-601a-46a2-9300-dee92489222b|2026-04-13 16:19:53.994311|CARD_00192|approved|1               |1                          |1                         |2                       |0.0                   |
|ba50776a-9faa-49fe-ab28-37fab8c8f70f|2026-04-13 16:16:22.131529|CARD_00062|approved|1              